# EDA · Bộ fine-tuning DataMate v2

Phân tích trực tiếp `data/fine_tuning_data_20261114_v2`: 3.665 train / 212 dev,
kiểm checksum và split, phân bố part A/B/C, kiểu đáp án, route, thinking, độ dài văn bản và mẫu minh họa.
Không cần GPU, model hoặc dữ liệu BTC khác. Cần `nbformat ipykernel pandas matplotlib`.

**Một lệnh từ repo** hoặc Run All trong Jupyter:

```bash
.venv-train/bin/python -u -c "import nbformat; n=nbformat.read('notebooks/01_eda.ipynb',4); exec('\n\n'.join(c.source for c in n.cells if c.cell_type=='code'))"
```

Output mặc định: `runs/eda_sft_v2/`. Số token trong phần cuối lấy từ báo cáo tokenizer đi kèm và được ghi rõ nguồn;
kiểm tra tokenizer hiện tại ở max length 16.384 bằng notebook train với `DATAMATE_MODE=check`.

In [ ]:
import os
import sys
from pathlib import Path

candidates = [Path(os.environ.get("DATAMATE_REPO", ".")).expanduser().resolve(), *Path.cwd().resolve().parents]
ROOT = next((p for p in candidates if (p / "training/train_lora_bf16.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Mở notebook trong checkout datamate hoặc đặt DATAMATE_REPO=/path/to/datamate_clone")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import hashlib
import json
import math
import random
import subprocess
import time

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from IPython import get_ipython
from IPython.display import Image, display

IN_JUPYTER = type(get_ipython()).__name__ == "ZMQInteractiveShell"


def path(value):
    p = Path(value).expanduser()
    return p if p.is_absolute() else ROOT / p


def show_table(frame):
    display(frame) if IN_JUPYTER else print(frame.to_string(index=False), flush=True)


def show_figure(fig, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(dest, dpi=150, bbox_inches="tight")
    if IN_JUPYTER:
        display(Image(filename=str(dest)))
    plt.close(fig)
    print(f"Figure: {dest}", flush=True)


def file_sha256(filename):
    h = hashlib.sha256()
    with open(filename, "rb") as handle:
        while chunk := handle.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()

In [ ]:
DATA_DIR = os.environ.get("DATA_DIR", "data/fine_tuning_data_20261114_v2")
OUT = os.environ.get("EDA_OUT", "runs/eda_sft_v2")

data_dir, out = path(DATA_DIR), path(OUT)
out.mkdir(parents=True, exist_ok=True)
print(f"Data: {data_dir}\nOutput: {out}", flush=True)

## 1. Tính toàn vẹn và chia train/dev

Kiểm tra dữ liệu hiện tại, không dùng số đếm hardcode để thay kết quả đọc file. Sai checksum/số lượng, trùng row id hoặc train/dev trùng câu hỏi/shop sẽ dừng. Các biến thể cùng question_id trong train được giữ nguyên.

In [ ]:
def audit_data(data_dir):
    """Read the supplied splits; never edit source rows or the bundle manifest."""
    manifest = json.loads((data_dir / "sft_manifest.json").read_text())
    for line in (data_dir / "SHA256SUMS").read_text().splitlines():
        if line.strip():
            expected, relative = line.split(maxsplit=1)
            filename = data_dir / relative.lstrip("*")
            if file_sha256(filename) != expected:
                raise ValueError(f"Checksum không khớp: {filename}")
    records, questions, shops, counts = [], {}, {}, {}
    allowed_roles = {"train": {"synthetic_train", "btc_train"}, "dev": {"synthetic_dev", "synthetic_train"}}
    for split, expected_rows in (("train", 3665), ("dev", 212)):
        filename = data_dir / f"{split}.jsonl"
        spec = manifest["splits"][split]
        if "sha256:" + file_sha256(filename) != spec["file_hash"]:
            raise ValueError(f"Không khớp sft_manifest.json: {filename}")
        questions[split], shops[split], row_ids = set(), set(), set()
        count = 0
        with filename.open(encoding="utf-8") as handle:
            for line in handle:
                if not line.strip():
                    continue
                row = json.loads(line)
                meta = row["metadata"]
                if meta.get("dataset_role") not in allowed_roles[split] or meta.get("split") != split or meta.get("training_eligible") is not True:
                    raise ValueError(f"Sai role/training_eligible: {split}, {row.get('id')}")
                if [m["role"] for m in row["prompt_messages"]] != ["system", "user"] or not row["completion"]:
                    raise ValueError(f"Sai prompt/completion: {row.get('id')}")
                if row["id"] in row_ids:
                    raise ValueError(f"Trùng row id trong {split}: {row['id']}")
                row_ids.add(row["id"])
                questions[split].add(meta["question_id"])
                shops[split].add(row["business_id"])
                records.append({"split": split, "id": row["id"], "question_id": meta["question_id"], "shop": row["business_id"],
                                "part": meta.get("part", "không ghi"), "dataset_role": meta["dataset_role"], "answer_type": meta.get("answer_type", "không ghi"),
                                "route": meta.get("route", "không ghi"), "thinking": bool(row.get("thinking_mode")),
                                "question": meta.get("question", ""), "completion": row["completion"],
                                "prompt_chars": sum(len(m["content"]) for m in row["prompt_messages"]),
                                "completion_chars": len(row["completion"]), "weak_group": meta.get("weak_group", "không ghi")})
                count += 1
        if count != expected_rows or count != spec["rows"]:
            raise ValueError(f"{split}: {count} mẫu, yêu cầu {expected_rows}")
        counts[split] = count
    if questions["train"] & questions["dev"] or shops["train"] & shops["dev"]:
        raise ValueError("Train/dev trùng question_id hoặc shop")
    print(f"Checksum OK | {counts['train']:,} train / {counts['dev']} dev | không trùng câu hỏi/shop", flush=True)
    return pd.DataFrame(records)

In [ ]:
data = audit_data(data_dir)
summary = data.groupby("split").agg(rows=("id", "size"), questions=("question_id", "nunique"), shops=("shop", "nunique"),
                                    thinking_rows=("thinking", "sum")).reset_index()
summary["thinking_share"] = summary["thinking_rows"] / summary["rows"]
show_table(summary)
summary.to_csv(out / "split_summary.csv", index=False)
selected = set(random.Random(3407).sample(range(len(data.loc[data["split"].eq("dev")])), 64))
dev = data.loc[data["split"].eq("dev")].reset_index(drop=True)
selected_dev = dev.iloc[sorted(selected)]
selected_dev[["id", "question_id", "shop", "part", "answer_type", "route"]].to_csv(out / "fixed_dev64.csv", index=False)
print(f"Dev64 dự kiến: {len(selected_dev)} mẫu, {selected_dev['shop'].nunique()} shop, seed 3407. "
      "Áp dụng sau kiểm tokenizer; notebook train yêu cầu cả 212 dev đều hợp lệ.", flush=True)

## 2. Thành phần dữ liệu

Số đếm lấy từ metadata từng dòng. `không ghi` nghĩa là trường metadata không được cung cấp, không tự suy đoán.

In [ ]:
fields = ("part", "answer_type", "route", "thinking")
print("Vai trò nguồn (split được xác định bằng file và metadata.split)", flush=True)
show_table(pd.crosstab(data["dataset_role"], data["split"]).reset_index())
fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
for field, ax in zip(fields, axes.flat):
    table = pd.crosstab(data[field].fillna("không ghi").astype(str), data["split"])
    print(f"\n{field}", flush=True)
    show_table(table.rename_axis(field).reset_index())
    table.to_csv(out / f"distribution_{field}.csv")
    table.plot.bar(ax=ax, rot=25 if field == "answer_type" else 0, title=field, ylabel="Rows")
show_figure(fig, out / "dataset_composition.png")
weak = pd.crosstab(data["weak_group"].fillna("không ghi"), data["split"])
print("\nWeak group (metadata)", flush=True)
show_table(weak.reset_index())

## 3. Độ dài văn bản

Đếm ký tự prompt/completion từ JSON hiện tại. Ký tự không phải token; không dùng các số này để đặt context length.

In [ ]:
lengths = data.groupby("split")[["prompt_chars", "completion_chars"]].describe(percentiles=[0.5, 0.9, 0.99])
show_table(lengths.stack(level=0, future_stack=True).reset_index())
lengths.to_csv(out / "character_lengths.csv")
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5), constrained_layout=True)
for field, ax in zip(("prompt_chars", "completion_chars"), axes):
    low, high = data[field].min(), data[field].max()
    bins = [low + (high - low) * i / 35 for i in range(36)]
    for split, group in data.groupby("split"):
        ax.hist(group[field], bins=bins, alpha=0.55, label=split)
    ax.set(title=field, xlabel="Characters", ylabel="Rows")
    ax.legend()
show_figure(fig, out / "character_lengths.png")

## 4. Tokenizer report đi kèm

Nguồn: `validation_report/data_report.json` của bundle. Đây là kết quả đã lưu, không phải phép tokenize mới trong notebook EDA; không dùng metadata `n_tokens` vốn thiếu ở một số dòng.

In [ ]:
saved_report_path = data_dir / "validation_report/data_report.json"
saved = json.loads(saved_report_path.read_text())
if any(saved[split]["rows_read"] != int(summary.set_index("split").loc[split, "rows"]) for split in ("train", "dev")):
    raise ValueError("Số dòng báo cáo tokenizer đi kèm không khớp dữ liệu")
rows = []
for split in ("train", "dev"):
    r = saved[split]
    rows.append({"split": split, "source": "bundled tokenizer report", "max_len": saved["max_len"], "rows_kept": r["rows_kept"],
                 "tokens_total": r["tokens_kept"], "prompt_plus_completion_p50": r["row_tokens"]["p50"],
                 "prompt_plus_completion_p99": r["row_tokens"]["p99"], "prompt_plus_completion_max": r["row_tokens"]["max"],
                 "completion_tokens_max": r["reply_tokens"]["max"], "label_share": r["label_share"], "rejected": str(r["rejected"])})
show_table(pd.DataFrame(rows))
print(f"Nguồn đã lưu: {saved_report_path}. Muốn xác nhận bằng tokenizer hiện tại: DATAMATE_MODE=check + lệnh notebook train.", flush=True)

## 5. Ví dụ từ dữ liệu hiện tại

Một mẫu của mỗi part/split: câu hỏi và phần đầu completion, không in toàn bộ rulebook/schema dài. Kiểm tra cấu trúc và token không thay thế đánh giá chất lượng nội dung/SQL.

In [ ]:
for (split, part), group in data.groupby(["split", "part"], dropna=False):
    example = group.iloc[0]
    print(f"\n[{split} / part {part}] {example['id']} | {example['answer_type']} | route={example['route']} | thinking={example['thinking']}")
    print("Question:", example["question"])
    print("Completion (first 650 characters):\n" + example["completion"][:650])
print(f"\nHoàn tất EDA. Bảng CSV và hình lưu tại {out}", flush=True)